# M1 — Baseline pencarian lintas modal SEA-VL

Notebook ini membangun baseline retrieval yang bisa dijalankan lokal dan dipakai ulang. Ia membaca kedua split, menyimpan embedding gambar, teks Indonesia, dan teks Inggris ke cache, lalu menghitung ranking serta metrik empat arah pada validasi.

Alur baseline:

    train.csv ──┐                         val.csv ──┐
                ├─ encoder beku ─ cache            ├─ encoder beku ─ cache
    images/ ────┘                                   └─ cosine similarity
                                                         ↓
                                             Top-K dan metrik empat arah

Semua encoder dibekukan. Notebook tidak melakukan pelatihan atau menambahkan transformasi pada embedding. Embedding train disediakan agar Peran 3 dapat menggunakannya kelak; angka baseline resmi dihitung hanya dari seluruh galeri validasi.

## Menjalankan di komputer lokal

1. Buka notebook dari folder proyek. Secara default, pencari lokasi akan menemukan bundle-mahasiswa di folder m0. Jika bundle disimpan di tempat lain, isi DATA_DIR pada sel konfigurasi.
2. Jalankan sel dari atas ke bawah. Untuk eksekusi pertama, pasang dependensi dalam environment Python notebook menggunakan perintah ini:

<pre>python -m pip install numpy pandas pillow torch sentence-transformers tqdm</pre>

3. Eksekusi pertama memerlukan internet untuk mengambil bobot model publik. Eksekusi berikutnya dapat memakai cache embedding yang sudah cocok.
4. Perangkat default adalah CPU, mengikuti syarat M1. Jalankan dari kernel Python 3.10 atau lebih baru.

Embedding cache dan hasil evaluasi akan dibuat di m1/outputs/baseline_pipeline_m1/.

In [ ]:
# Pengaturan yang biasanya perlu ditinjau atau diubah.
from pathlib import Path

DATA_DIR = None                 # Contoh: Path("/lokasi/bundle-mahasiswa") atau path menuju val.csv
DEVICE = "cpu"                  # Default M1: "cpu". Ubah ke "cuda" hanya jika PyTorch CUDA tersedia.
SEED = 42
IMAGE_BATCH_SIZE = 16
TEXT_BATCH_SIZE = 32
TOP_K = 10
RECALL_KS = (1, 5, 10)
MAP_K = 10

# Nama model dan revision dipatok agar baseline dapat direproduksi.
MODEL_IMAGE_ENGLISH = "sentence-transformers/clip-ViT-B-32"
MODEL_TEXT_INDONESIAN = "sentence-transformers/clip-ViT-B-32-multilingual-v1"
REVISION_IMAGE_ENGLISH = "327ab6726d33c0e22f920c83f2ff9e4bd38ca37f"
REVISION_TEXT_INDONESIAN = "58edf8cada9e398793dca955574a48cbb7f18be2"

In [15]:
import sys
!{sys.executable} -m pip install numpy pandas torch pillow tqdm sentence-transformers

  Using cached scikit_learn-1.9.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 74.8 kB/s  0:00:09 81.9 kB/s eta 0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 66.5 kB/s  0:00:11 66.4 kB/s eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.4 kB/s  0:00:43 81.2 kB/s eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 M

In [11]:
import hashlib
import json
import os
import time
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm

def locate_bundle(configured_path=None):
    """Cari folder yang berisi train.csv, val.csv, dan images/."""
    if configured_path is not None:
        path = Path(configured_path).expanduser().resolve()
        root = path.parent if path.name.lower() == "val.csv" else path
        required = [root / "train.csv", root / "val.csv", root / "images"]
        missing = [str(item) for item in required if not item.exists()]
        if missing:
            raise FileNotFoundError("Bundle tidak lengkap. Tidak ditemukan: " + ", ".join(missing))
        return root

    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base / "bundle-mahasiswa",
            base / "m0" / "bundle-mahasiswa",
        ])
    found = sorted({p.resolve() for p in candidates
                    if (p / "train.csv").is_file()
                    and (p / "val.csv").is_file()
                    and (p / "images").is_dir()}, key=str)
    if not found:
        raise FileNotFoundError(
            "Folder bundle-mahasiswa tidak ditemukan. Isi DATA_DIR dengan path bundle "
            "yang memuat train.csv, val.csv, dan images/."
        )
    if len(found) > 1:
        raise ValueError(
            "Ada lebih dari satu bundle. Tentukan DATA_DIR secara eksplisit: "
            + ", ".join(map(str, found))
        )
    return found[0]

def locate_m1_folder():
    """Pilih folder m1 proyek sebagai lokasi output, berdasarkan working directory."""
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == "m1":
        return cwd
    if (cwd / "m1").is_dir():
        return cwd / "m1"
    for parent in cwd.parents:
        if parent.name.lower() == "m1":
            return parent
        if (parent / "m1").is_dir():
            return parent / "m1"
    return cwd / "m1"

DATA_ROOT = locate_bundle(DATA_DIR)
M1_DIR = locate_m1_folder()
OUTPUT_DIR = M1_DIR / "outputs" / "baseline_pipeline_m1"
CACHE_DIR = OUTPUT_DIR / "cache_embeddings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Bundle :", DATA_ROOT)
print("Output :", OUTPUT_DIR)
print("Device :", DEVICE)

Bundle : /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/bundle-mahasiswa
Output : /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/m1/outputs/baseline_pipeline_m1
Device : cpu


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Membaca data dan menjaga pasangan

Setiap baris manifest memasangkan satu image_id, file gambar, caption Indonesia, dan caption Inggris. Notebook mempertahankan urutan baris CSV agar indeks pasangan ground-truth konsisten antara embedding dan evaluasi.

train.csv digunakan untuk membuat cache fitur dan contoh pencarian kualitatif. val.csv digunakan penuh untuk metrik baseline dan daftar prediksi Top-K. Tidak ada parameter yang dipilih dari hasil validasi.

Pemeriksaan ID dan file yang sama pada train/validasi menghentikan run bila ada tumpang tindih. Caption duplikat hanya dilaporkan; tidak ada baris yang dihapus diam-diam.

In [12]:
REQUIRED_COLUMNS = ["image_id", "file", "caption_id", "caption_en"]

def read_manifest(csv_path, expected_split):
    frame = pd.read_csv(csv_path, dtype={"image_id": str, "file": str})
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"{csv_path.name}: kolom wajib tidak ada: {missing}")
    if frame.empty:
        raise ValueError(f"{csv_path.name} kosong.")
    for column in REQUIRED_COLUMNS:
        if frame[column].isna().any() or frame[column].astype(str).str.strip().eq("").any():
            raise ValueError(f"{csv_path.name}: kolom {column} memiliki nilai kosong.")
    for column in ["image_id", "file"]:
        if frame[column].duplicated().any():
            raise ValueError(f"{csv_path.name}: nilai {column} harus unik.")
    if "split" in frame.columns and not frame["split"].astype(str).eq(expected_split).all():
        raise ValueError(f"{csv_path.name}: label split tidak seluruhnya {expected_split!r}.")
    return frame.reset_index(drop=True)

def resolve_image_path(bundle_root, relative_path):
    relative = Path(str(relative_path))
    if relative.is_absolute():
        raise ValueError(f"Path gambar harus relatif terhadap bundle: {relative}")
    resolved = (bundle_root / relative).resolve()
    try:
        resolved.relative_to(bundle_root.resolve())
    except ValueError as exc:
        raise ValueError(f"Path gambar keluar dari folder bundle: {relative}") from exc
    if not resolved.is_file():
        raise FileNotFoundError(f"Gambar tidak ditemukan: {resolved}")
    return resolved

train_df = read_manifest(DATA_ROOT / "train.csv", "train")
val_df = read_manifest(DATA_ROOT / "val.csv", "val")
train_paths = [resolve_image_path(DATA_ROOT, item) for item in train_df["file"]]
val_paths = [resolve_image_path(DATA_ROOT, item) for item in val_df["file"]]

overlap_ids = set(train_df["image_id"]) & set(val_df["image_id"])
overlap_files = set(train_df["file"]) & set(val_df["file"])
if overlap_ids or overlap_files:
    raise ValueError(
        f"Ditemukan tumpang tindih train/val: {len(overlap_ids)} image_id dan "
        f"{len(overlap_files)} file. Tinjau bundle bersama Peran 1."
    )

audit_table = pd.DataFrame([
    {
        "split": "train",
        "jumlah_baris": len(train_df),
        "caption_ID_duplikat": int(train_df["caption_id"].duplicated().sum()),
        "caption_EN_duplikat": int(train_df["caption_en"].duplicated().sum()),
    },
    {
        "split": "val",
        "jumlah_baris": len(val_df),
        "caption_ID_duplikat": int(val_df["caption_id"].duplicated().sum()),
        "caption_EN_duplikat": int(val_df["caption_en"].duplicated().sum()),
    },
])
display(audit_table)
print(f"Train/val berbagi ID: {len(overlap_ids)} | file: {len(overlap_files)}")
print(f"Validasi memuat {len(val_df)} pasangan; R@1 acak sebagai acuan = {1 / len(val_df):.4%}.")

,split,jumlah_baris,caption_ID_duplikat,caption_EN_duplikat
0,train,2072,47,48
1,val,275,1,1


Train/val berbagi ID: 0 | file: 0
Validasi memuat 275 pasangan; R@1 acak sebagai acuan = 0.3636%.


## Encoder yang digunakan

- Gambar dan caption Inggris memakai model CLIP yang sama, sesuai contoh baseline TOR.
- Caption Indonesia memakai encoder CLIP multilingual.
- Output tiap encoder dinormalisasi dengan panjang L2 satu. Karena itu, perkalian dot product antarembedding sama dengan cosine similarity.
- Model berada pada mode evaluasi dan semua bobot dibekukan. Notebook tidak mengubah bobot pretrained.

In [13]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if DEVICE not in ("cpu", "cuda"):
    raise ValueError('DEVICE harus "cpu" atau "cuda".')
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("PyTorch CUDA tidak tersedia. Gunakan DEVICE='cpu'.")
if IMAGE_BATCH_SIZE < 1 or TEXT_BATCH_SIZE < 1:
    raise ValueError("Ukuran batch harus minimal 1.")

if DEVICE == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

_IMAGE_MODEL = None
_TEXT_ID_MODEL = None

def get_models():
    """Muat encoder hanya saat cache perlu dibuat atau query baru dimasukkan."""
    global _IMAGE_MODEL, _TEXT_ID_MODEL
    if _IMAGE_MODEL is None or _TEXT_ID_MODEL is None:
        from sentence_transformers import SentenceTransformer
        print("Memuat model publik. Unduhan awal memerlukan koneksi internet.")
        _IMAGE_MODEL = SentenceTransformer(
            MODEL_IMAGE_ENGLISH,
            revision=REVISION_IMAGE_ENGLISH,
            device=DEVICE,
        )
        _TEXT_ID_MODEL = SentenceTransformer(
            MODEL_TEXT_INDONESIAN,
            revision=REVISION_TEXT_INDONESIAN,
            device=DEVICE,
        )
        for model in (_IMAGE_MODEL, _TEXT_ID_MODEL):
            model.eval()
            model.requires_grad_(False)
        trainable = sum(
            parameter.numel()
            for model in (_IMAGE_MODEL, _TEXT_ID_MODEL)
            for parameter in model.parameters()
            if parameter.requires_grad
        )
        if trainable != 0:
            raise RuntimeError(f"Ditemukan {trainable} parameter encoder yang masih dapat dilatih.")
    return _IMAGE_MODEL, _TEXT_ID_MODEL

def installed_versions():
    result = {}
    for package in ["numpy", "pandas", "Pillow", "torch", "sentence-transformers", "transformers"]:
        try:
            result[package] = version(package)
        except PackageNotFoundError:
            result[package] = "not-installed"
    return result

print("Encoder siap dimuat secara lazy (hanya bila cache belum tersedia).")
print("Versi paket:", installed_versions())

Encoder siap dimuat secara lazy (hanya bila cache belum tersedia).
Versi paket: {'numpy': '2.5.4', 'pandas': '3.1.1', 'Pillow': '12.3.0', 'torch': '2.15.0', 'sentence-transformers': 'not-installed', 'transformers': 'not-installed'}


## Membuat dan memakai cache embedding

Cache dibuat terpisah untuk train dan validasi. Setiap berkas memuat tiga matriks berdimensi N × 512, urutan image_id, dan file. Nama cache bergantung pada hash CSV, isi file gambar, revisi model, versi paket, dan konfigurasi. Cache lama tidak dipakai jika salah satu komponennya berubah.

Dengan begitu, Peran 3 dapat memuat fitur train dari NPZ untuk eksperimen berikutnya dan tetap mengevaluasi pada cache val yang sama.

In [16]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def normalize_rows(embeddings):
    values = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    if np.any(norms == 0) or not np.isfinite(values).all():
        raise ValueError("Embedding berisi nilai non-finite atau vektor nol.")
    return values / norms

def validate_embeddings(bundle, frame):
    expected_shape = (len(frame), 512)
    if not np.array_equal(bundle["image_ids"], frame["image_id"].to_numpy(dtype=str)):
        raise ValueError("Urutan image_id pada embedding tidak cocok dengan manifest.")
    if not np.array_equal(bundle["files"], frame["file"].to_numpy(dtype=str)):
        raise ValueError("Urutan file pada embedding tidak cocok dengan manifest.")
    for key in ["E", "T_ID", "T_EN"]:
        values = bundle[key]
        if values.shape != expected_shape or not np.isfinite(values).all():
            raise ValueError(f"Embedding {key} punya bentuk/nilai tidak valid: {values.shape}.")
        if not np.allclose(np.linalg.norm(values, axis=1), 1.0, atol=1e-4):
            raise ValueError(f"Embedding {key} tidak ternormalisasi.")

def fingerprint_split(split_name, frame, paths):
    """Hash CSV dan konten gambar agar perubahan data membatalkan cache lama."""
    csv_path = DATA_ROOT / f"{split_name}.csv"
    image_hashes = [
        (str(image_id), str(relative_file), sha256_file(path))
        for image_id, relative_file, path
        in zip(frame["image_id"], frame["file"], paths)
    ]
    payload = {
        "csv_sha256": sha256_file(csv_path),
        "images": image_hashes,
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()

def make_cache_spec(split_name, frame, paths):
    return {
        "pipeline_version": "m1_frozen_retrieval_v1",
        "split": split_name,
        "data_fingerprint": fingerprint_split(split_name, frame, paths),
        "image_model": MODEL_IMAGE_ENGLISH,
        "image_revision": REVISION_IMAGE_ENGLISH,
        "text_id_model": MODEL_TEXT_INDONESIAN,
        "text_id_revision": REVISION_TEXT_INDONESIAN,
        "text_en_model": MODEL_IMAGE_ENGLISH,
        "text_en_revision": REVISION_IMAGE_ENGLISH,
        "device": DEVICE,
        "image_batch_size": IMAGE_BATCH_SIZE,
        "text_batch_size": TEXT_BATCH_SIZE,
        "embedding_dtype": "float32",
        "packages": installed_versions(),
    }

def cache_paths(split_name, spec):
    key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    return CACHE_DIR / f"{split_name}_{key}.npz", CACHE_DIR / f"{split_name}_{key}.json"

def load_cache_if_valid(npz_path, json_path, spec, frame):
    if not npz_path.is_file() or not json_path.is_file():
        return None
    try:
        metadata = json.loads(json_path.read_text(encoding="utf-8"))
        if metadata.get("spec") != spec:
            return None
        if sha256_file(npz_path) != metadata.get("embedding_cache_sha256"):
            print(f"Cache {npz_path.name} rusak; embedding akan dibuat ulang.")
            return None
        with np.load(npz_path, allow_pickle=False) as saved:
            bundle = {key: saved[key].copy()
                      for key in ["E", "T_ID", "T_EN", "image_ids", "files"]}
        validate_embeddings(bundle, frame)
        return bundle
    except Exception as error:
        print(f"Cache {npz_path.name} tidak dapat dipakai ({error}); akan dibuat ulang.")
        return None

def encode_images(image_model, image_paths):
    blocks = []
    for start in tqdm(range(0, len(image_paths), IMAGE_BATCH_SIZE), desc="Encoding gambar"):
        opened = []
        try:
            for path in image_paths[start:start + IMAGE_BATCH_SIZE]:
                with Image.open(path) as image:
                    opened.append(image.convert("RGB"))
            with torch.inference_mode():
                block = image_model.encode(
                    opened,
                    batch_size=IMAGE_BATCH_SIZE,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                )
            blocks.append(normalize_rows(block))
        finally:
            for image in opened:
                image.close()
    return np.concatenate(blocks, axis=0)

def encode_texts(model, texts, batch_size, label):
    with torch.inference_mode():
        values = model.encode(
            list(texts),
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
    return normalize_rows(values)

def load_or_encode_split(split_name, frame, image_paths):
    spec = make_cache_spec(split_name, frame, image_paths)
    npz_path, json_path = cache_paths(split_name, spec)
    cached = load_cache_if_valid(npz_path, json_path, spec, frame)
    if cached is not None:
        print(f"{split_name}: memakai cache cocok ({npz_path.name}).")
        return cached, {"cache_file": str(npz_path), "cache_used": True, "spec": spec}

    image_model, text_id_model = get_models()
    started = time.perf_counter()
    embeddings = {
        "E": encode_images(image_model, image_paths),
        "T_ID": encode_texts(text_id_model, frame["caption_id"], TEXT_BATCH_SIZE,
                             f"Teks ID {split_name}"),
        "T_EN": encode_texts(image_model, frame["caption_en"], TEXT_BATCH_SIZE,
                             f"Teks EN {split_name}"),
        "image_ids": frame["image_id"].to_numpy(dtype=str),
        "files": frame["file"].to_numpy(dtype=str),
    }
    validate_embeddings(embeddings, frame)
    np.savez_compressed(npz_path, **embeddings)
    metadata = {
        "spec": spec,
        "created_at_jakarta": datetime.now(ZoneInfo("Asia/Jakarta")).isoformat(),
        "rows": len(frame),
        "embedding_shape": list(embeddings["E"].shape),
        "encoding_seconds": time.perf_counter() - started,
        "trainable_encoder_parameters": 0,
        "embedding_cache_sha256": sha256_file(npz_path),
    }
    json_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{split_name}: embedding tersimpan ke {npz_path}")
    return embeddings, {"cache_file": str(npz_path), "cache_used": False, "spec": spec}

run_started = time.perf_counter()
train_embeddings, train_cache_info = load_or_encode_split("train", train_df, train_paths)
val_embeddings, val_cache_info = load_or_encode_split("val", val_df, val_paths)

print("Bentuk E / T_ID / T_EN:")
for split_name, bundle in [("train", train_embeddings), ("val", val_embeddings)]:
    print(f"  {split_name:5s}", bundle["E"].shape, bundle["T_ID"].shape, bundle["T_EN"].shape)
print("Semua encoder beku; parameter encoder yang dapat dilatih = 0.")

W1112 20:07:19.495000 13904 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Memuat model publik. Unduhan awal memerlukan koneksi internet.


Batches: 100%|██████████| 65/65 [00:04<00:00, 13.79it/s]


train: embedding tersimpan ke /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/m1/outputs/baseline_pipeline_m1/cache_embeddings/train_9e350cdf3477e6c0.npz


Batches: 100%|██████████| 9/9 [00:00<00:00, 13.39it/s]


val: embedding tersimpan ke /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/m1/outputs/baseline_pipeline_m1/cache_embeddings/val_82e5b0dd91f26904.npz
Bentuk E / T_ID / T_EN:
  train (2072, 512) (2072, 512) (2072, 512)
  val   (275, 512) (275, 512) (275, 512)
Semua encoder beku; parameter encoder yang dapat dilatih = 0.


## Cosine similarity dan evaluasi empat arah

Untuk tiap bahasa, satu matriks skor dibentuk dengan ukuran jumlah gambar × jumlah caption. Transpose matriks memberi arah pencarian balik:

    E_val @ T_ID_val.T  → image ke teks Indonesia
    (E_val @ T_ID_val.T).T  → teks Indonesia ke image

Pencarian Inggris memakai pasangan yang sama dengan T_EN. Metrik memakai indeks pasangan pada manifest sebagai satu pasangan relevan per query: Recall@K dan mAP@10. Dalam skenario satu pasangan relevan, AP@10 adalah kebalikan rank pasangan bila masuk 10 teratas, atau nol jika tidak.

Skor similarity adalah cosine, bukan probabilitas. Urutan skor menentukan ranking; tie diurutkan menurut urutan kandidat pada CSV.

In [17]:
def cosine_similarity_matrix(query_embeddings, candidate_embeddings):
    """Kembalikan matriks query × gallery; input embedding diasumsikan ternormalisasi."""
    queries = np.asarray(query_embeddings, dtype=np.float32)
    candidates = np.asarray(candidate_embeddings, dtype=np.float32)
    if queries.ndim != 2 or candidates.ndim != 2:
        raise ValueError("Embedding query dan kandidat harus berupa matriks 2D.")
    if queries.shape[1] != candidates.shape[1]:
        raise ValueError("Dimensi embedding query dan kandidat harus sama.")
    return queries @ candidates.T

def rank_top_k(scores, k=TOP_K):
    """Indeks kandidat teratas per query; tie stabil mengikuti urutan galeri."""
    matrix = np.asarray(scores)
    if matrix.ndim != 2 or matrix.shape[1] == 0:
        raise ValueError("Scores harus berupa matriks 2D dengan kandidat.")
    if k < 1:
        raise ValueError("k harus minimal 1.")
    return np.argsort(-matrix, axis=1, kind="stable")[:, :min(k, matrix.shape[1])]

def relevant_gallery_indices(query_manifest, gallery_manifest):
    """Pasangkan query dan kandidat berdasarkan image_id pada manifest."""
    lookup = {image_id: index
              for index, image_id in enumerate(gallery_manifest["image_id"].astype(str))}
    try:
        return np.asarray([
            lookup[image_id] for image_id in query_manifest["image_id"].astype(str)
        ], dtype=np.int64)
    except KeyError as error:
        raise ValueError(f"Pasangan ground-truth tidak ada di galeri: {error}") from error

def ranks_of_relevant_items(scores, relevant_indices):
    matrix = np.asarray(scores)
    relevant = np.asarray(relevant_indices, dtype=np.int64)
    if matrix.shape[0] != len(relevant):
        raise ValueError("Jumlah query dan indeks pasangan relevan berbeda.")
    if np.any(relevant < 0) or np.any(relevant >= matrix.shape[1]):
        raise ValueError("Indeks pasangan relevan berada di luar galeri.")
    order = np.argsort(-matrix, axis=1, kind="stable")
    return np.argmax(order == relevant[:, None], axis=1) + 1

def evaluate_retrieval(scores, relevant_indices, recall_ks=RECALL_KS, map_k=MAP_K):
    """Metrik untuk satu item relevan per query sesuai pasangan manifest."""
    ranks = ranks_of_relevant_items(scores, relevant_indices)
    result = {f"Recall@{k}": float(np.mean(ranks <= min(k, scores.shape[1])))
              for k in recall_ks}
    clipped_map_k = min(map_k, scores.shape[1])
    reciprocal_rank = np.where(ranks <= clipped_map_k, 1.0 / ranks, 0.0)
    result[f"mAP@{map_k}"] = float(np.mean(reciprocal_rank))
    return result

def build_topk_table(scores, query_manifest, gallery_manifest,
                     direction, query_kind, k=TOP_K):
    """Bentuk tabel terurut yang dapat disimpan dan digunakan ulang."""
    order = rank_top_k(scores, k)
    relevant = relevant_gallery_indices(query_manifest, gallery_manifest)
    ranks = ranks_of_relevant_items(scores, relevant)
    rows = []
    for query_index, candidate_indices in enumerate(order):
        query = query_manifest.iloc[query_index]
        for rank, candidate_index in enumerate(candidate_indices, start=1):
            candidate = gallery_manifest.iloc[int(candidate_index)]
            rows.append({
                "direction": direction,
                "query_rank": rank,
                "query_id": str(query["image_id"]),
                "query_kind": query_kind,
                "query_text": query["caption_id"] if query_kind == "text_id"
                              else query["caption_en"] if query_kind == "text_en" else "",
                "query_file": query["file"] if query_kind == "image" else "",
                "candidate_id": str(candidate["image_id"]),
                "candidate_file": candidate["file"],
                "candidate_caption_id": candidate["caption_id"],
                "candidate_caption_en": candidate["caption_en"],
                "cosine_similarity": float(scores[query_index, candidate_index]),
                "rank_of_manifest_pair": int(ranks[query_index]),
                "is_manifest_pair": bool(candidate_index == relevant[query_index]),
            })
    return pd.DataFrame(rows)

val_E = val_embeddings["E"]
val_T_ID = val_embeddings["T_ID"]
val_T_EN = val_embeddings["T_EN"]
S_ID = cosine_similarity_matrix(val_E, val_T_ID)
S_EN = cosine_similarity_matrix(val_E, val_T_EN)

direction_specs = {
    "image_to_text_id": (S_ID, val_df, val_df, "image"),
    "text_id_to_image": (S_ID.T, val_df, val_df, "text_id"),
    "image_to_text_en": (S_EN, val_df, val_df, "image"),
    "text_en_to_image": (S_EN.T, val_df, val_df, "text_en"),
}

metric_rows = []
val_topk_tables = {}
for direction, (scores, queries, gallery, query_kind) in direction_specs.items():
    relevant = relevant_gallery_indices(queries, gallery)
    metric_rows.append({
        "split": "val",
        "direction": direction,
        "n_queries": scores.shape[0],
        "n_candidates": scores.shape[1],
        "seed": SEED,
        **evaluate_retrieval(scores, relevant),
    })
    val_topk_tables[direction] = build_topk_table(
        scores, queries, gallery, direction, query_kind, k=TOP_K
    )

metrics_val = pd.DataFrame(metric_rows)
display(metrics_val.round(6))
print("Metrik dihitung atas seluruh baris val.csv, tanpa sampling.")

,split,direction,n_queries,n_candidates,seed,Recall@1,Recall@5,Recall@10,mAP@10
0,val,image_to_text_id,275,275,42,0.214545,0.378182,0.505455,0.293206
1,val,text_id_to_image,275,275,42,0.196364,0.414545,0.501818,0.283051
2,val,image_to_text_en,275,275,42,0.443636,0.712727,0.832727,0.557912
3,val,text_en_to_image,275,275,42,0.378182,0.676364,0.770909,0.505645


Metrik dihitung atas seluruh baris val.csv, tanpa sampling.


In [18]:
# Selisih R@1 EN - ID pada arah retrieval yang sama, dalam poin persentase.
metric_lookup = metrics_val.set_index("direction")
language_gap = pd.DataFrame([
    {
        "direction": "image_to_text",
        "R@1_ID": metric_lookup.loc["image_to_text_id", "Recall@1"],
        "R@1_EN": metric_lookup.loc["image_to_text_en", "Recall@1"],
        "gap_EN_minus_ID_percentage_points": 100 * (
            metric_lookup.loc["image_to_text_en", "Recall@1"]
            - metric_lookup.loc["image_to_text_id", "Recall@1"]
        ),
    },
    {
        "direction": "text_to_image",
        "R@1_ID": metric_lookup.loc["text_id_to_image", "Recall@1"],
        "R@1_EN": metric_lookup.loc["text_en_to_image", "Recall@1"],
        "gap_EN_minus_ID_percentage_points": 100 * (
            metric_lookup.loc["text_en_to_image", "Recall@1"]
            - metric_lookup.loc["text_id_to_image", "Recall@1"]
        ),
    },
])
display(language_gap.round(3))
print("Tanda positif berarti R@1 Inggris lebih tinggi pada arah itu; hitungannya mengikuti hasil aktual.")

,direction,R@1_ID,R@1_EN,gap_EN_minus_ID_percentage_points
0,image_to_text,0.215,0.444,22.909
1,text_to_image,0.196,0.378,18.182


Tanda positif berarti R@1 Inggris lebih tinggi pada arah itu; hitungannya mengikuti hasil aktual.


## Prediksi Top-K dan contoh kueri Indonesia

Notebook menyimpan ranking Top-K untuk setiap query validasi pada empat arah. Berkas tersebut mencakup skor cosine, urutan kandidat, dan rank pasangan manifest; tidak ada query validasi yang dipilih manual untuk dibahas.

Untuk contoh kualitatif M1, sel berikut memilih sampai dua kegagalan teks-ID ke gambar dari train.csv, lalu melengkapinya sampai sepuluh query berdasarkan urutan CSV. Tabel menampilkan tiga kandidat pertama per query. Jika train tidak memiliki dua kegagalan, jumlah yang tersedia dilaporkan apa adanya.

In [19]:
train_scores_id = cosine_similarity_matrix(
    train_embeddings["T_ID"], train_embeddings["E"]
)
train_rank_id = ranks_of_relevant_items(
    train_scores_id, relevant_gallery_indices(train_df, train_df)
)
train_topk_id = build_topk_table(
    train_scores_id,
    train_df,
    train_df,
    direction="text_id_to_image",
    query_kind="text_id",
    k=TOP_K,
)

failure_indices = np.flatnonzero(train_rank_id > 1).tolist()
selected_indices = failure_indices[:2]
for index in range(len(train_df)):
    if len(selected_indices) >= min(10, len(train_df)):
        break
    if index not in selected_indices:
        selected_indices.append(index)

selected_ids = set(train_df.iloc[selected_indices]["image_id"].astype(str))
examples_train = train_topk_id[
    train_topk_id["query_id"].isin(selected_ids)
    & (train_topk_id["query_rank"] <= 3)
].copy()

display(examples_train[
    ["query_id", "query_text", "query_rank", "candidate_id",
     "candidate_caption_id", "cosine_similarity",
     "rank_of_manifest_pair", "is_manifest_pair"]
].sort_values(["query_id", "query_rank"]).reset_index(drop=True))

print(f"Contoh query train ditampilkan: {len(selected_indices)}.")
print(f"Kegagalan R@1 nyata yang tersedia pada train: {len(failure_indices)}.")
if len(failure_indices) < 2:
    print("Train memiliki kurang dari dua kegagalan; tidak ada contoh gagal yang dibuat-buat.")

,query_id,query_text,query_rank,candidate_id,candidate_caption_id,cosine_similarity,rank_of_manifest_pair,is_manifest_pair
0,svl_140083991,Nasi padang pakai ayam panggang,1,svl_147525916,Rengginang adalah sejenis kerupuk tebal yang t...,0.303640,148,False
1,svl_140083991,Nasi padang pakai ayam panggang,2,svl_147525683,Nasi uduk (nasi putih yang dimasak dengan sant...,0.297675,148,False
2,svl_140083991,Nasi padang pakai ayam panggang,3,svl_142113024,"Foto dari Nasi Padang, salah satu makanan trad...",0.292374,148,False
3,svl_140083992,Nasi goreng dengan telur ceplok,1,svl_147563090,"Menggoreng Pisang untuk membuat pisang goreng,...",0.298657,28,False
4,svl_140083992,Nasi goreng dengan telur ceplok,2,svl_142113344,"Seblak tanpa kuah, terbuat dari kerupuk dan telur",0.295491,28,False
5,svl_140083992,Nasi goreng dengan telur ceplok,3,svl_144425127,Ini cuanki dengan indomie khas Bandung,0.292561,28,False
6,svl_140083994,"Dua porsi Mie Bangka, masing-masing dengan sup...",1,svl_140084164,Es kembang tahu dengan es serut dan kuah jahe ...,0.329053,2,False
7,svl_140083994,"Dua porsi Mie Bangka, masing-masing dengan sup...",2,svl_140083994,"Dua porsi Mie Bangka, masing-masing dengan sup...",0.328417,2,True
8,svl_140083994,"Dua porsi Mie Bangka, masing-masing dengan sup...",3,svl_147525741,Dua mangkok bakwan Malang,0.326007,2,False
9,svl_140083997,nasi goreng dengan telur dan sate,1,svl_144924022,Nasi telur pedas dengan kremes,0.306439,209,False


Contoh query train ditampilkan: 10.
Kegagalan R@1 nyata yang tersedia pada train: 1932.


## Memakai pipeline untuk query baru

Fungsi berikut mengubah satu teks baru menjadi embedding menggunakan encoder yang sesuai, lalu mencari Top-K gambar di galeri validasi. Untuk teks Indonesia gunakan language="id"; untuk teks Inggris gunakan language="en". Hasilnya berupa DataFrame agar mudah diperiksa atau diekspor.

Contoh pemakaian setelah Run All:

    retrieve_text_query("Penari topeng Cirebon berkostum merah", language="id", k=5)

Fungsi cosine_similarity_matrix, rank_top_k, evaluate_retrieval, dan build_topk_table juga dapat dipakai Peran 3 saat menghubungkan pipeline ini dengan komponen eksperimen mereka.

In [20]:
def retrieve_text_query(query_text, language="id", k=TOP_K):
    """Cari gambar validasi untuk satu query teks baru berbahasa ID atau EN."""
    if language not in ("id", "en"):
        raise ValueError('language harus "id" atau "en".')
    image_model, text_id_model = get_models()
    encoder = text_id_model if language == "id" else image_model
    with torch.inference_mode():
        query_embedding = encoder.encode(
            [query_text],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    query_embedding = normalize_rows(query_embedding)
    scores = cosine_similarity_matrix(query_embedding, val_E)[0]
    indices = rank_top_k(scores[None, :], k=k)[0]
    return pd.DataFrame([
        {
            "rank": rank,
            "image_id": str(val_df.iloc[index]["image_id"]),
            "file": val_df.iloc[index]["file"],
            "caption_id": val_df.iloc[index]["caption_id"],
            "caption_en": val_df.iloc[index]["caption_en"],
            "cosine_similarity": float(scores[index]),
        }
        for rank, index in enumerate(indices, start=1)
    ])

def retrieve_image_query(image_path, language="id", k=TOP_K):
    """Cari caption validasi berbahasa ID atau EN untuk satu gambar baru."""
    if language not in ("id", "en"):
        raise ValueError('language harus "id" atau "en".')
    image_model, _ = get_models()
    with Image.open(image_path) as image:
        rgb = image.convert("RGB")
    try:
        with torch.inference_mode():
            query_embedding = image_model.encode(
                [rgb],
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
    finally:
        rgb.close()
    query_embedding = normalize_rows(query_embedding)
    gallery = val_T_ID if language == "id" else val_T_EN
    scores = cosine_similarity_matrix(query_embedding, gallery)[0]
    indices = rank_top_k(scores[None, :], k=k)[0]
    return pd.DataFrame([
        {
            "rank": rank,
            "image_id": str(val_df.iloc[index]["image_id"]),
            "caption": val_df.iloc[index]["caption_id"] if language == "id"
                       else val_df.iloc[index]["caption_en"],
            "cosine_similarity": float(scores[index]),
        }
        for rank, index in enumerate(indices, start=1)
    ])

# Jalankan contoh ini sendiri bila ingin mencoba query baru:
# display(retrieve_text_query("Rumah gadang khas Minangkabau", language="id", k=5))

## Menyimpan hasil run

Output utama:
- cache NPZ train dan val dengan metadata pasangan,
- tabel metrik validasi empat arah,
- language gap R@1,
- ranking Top-K seluruh query validasi di empat arah,
- contoh query Indonesia yang dipilih hanya dari train,
- metadata run dan konfigurasi model/data.

CSV metrik validasi memakai rasio 0–1. Kolom cosine_similarity juga bukan probabilitas.

In [21]:
# Simpan tabel hasil yang mudah dibaca ulang oleh notebook berikutnya.
metrics_path = OUTPUT_DIR / "metrics_val_four_directions.csv"
gap_path = OUTPUT_DIR / "language_gap_recall_at_1.csv"
examples_path = OUTPUT_DIR / "examples_train_text_id_to_image_topk.csv"
predictions_path = OUTPUT_DIR / "predictions_val_topk_four_directions.csv"

metrics_val.to_csv(metrics_path, index=False)
language_gap.to_csv(gap_path, index=False)
examples_train.to_csv(examples_path, index=False)
pd.concat(val_topk_tables.values(), ignore_index=True).to_csv(predictions_path, index=False)

def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Jenis tidak dapat ditulis ke JSON: {type(value).__name__}")

run_metadata = {
    "created_at_jakarta": datetime.now(ZoneInfo("Asia/Jakarta")).isoformat(),
    "data_root": str(DATA_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "device": DEVICE,
    "seed": SEED,
    "packages": installed_versions(),
    "models": {
        "image_and_english_text": MODEL_IMAGE_ENGLISH,
        "image_and_english_text_revision": REVISION_IMAGE_ENGLISH,
        "indonesian_text": MODEL_TEXT_INDONESIAN,
        "indonesian_text_revision": REVISION_TEXT_INDONESIAN,
    },
    "split_rows": {"train": len(train_df), "val": len(val_df)},
    "cache": {
        "train": train_cache_info,
        "val": val_cache_info,
    },
    "evaluation": {
        "split": "val",
        "gallery": "all rows in val.csv, original CSV order",
        "ground_truth": "paired row/image_id from manifest; one relevant item per query",
        "directions": list(direction_specs),
        "recall_ks": list(RECALL_KS),
        "map_k": MAP_K,
        "top_k": TOP_K,
        "tie_break": "stable; candidate CSV order",
        "encoder_trainable_parameters": 0,
    },
    "artifacts": {
        "metrics": str(metrics_path),
        "language_gap": str(gap_path),
        "examples_train": str(examples_path),
        "predictions_val_topk": str(predictions_path),
    },
}
metadata_path = OUTPUT_DIR / "run_metadata.json"
metadata_path.write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False, default=json_safe),
    encoding="utf-8",
)

print("Hasil tersimpan di:", OUTPUT_DIR)
for artifact in [metrics_path, gap_path, examples_path, predictions_path, metadata_path]:
    print(" -", artifact.name)
print("Cache train:", train_cache_info["cache_file"])
print("Cache val  :", val_cache_info["cache_file"])
print(f"Waktu pipeline sejak sel encoding dimulai: {time.perf_counter() - run_started:.1f} detik.")

Hasil tersimpan di: /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/m1/outputs/baseline_pipeline_m1
 - metrics_val_four_directions.csv
 - language_gap_recall_at_1.csv
 - examples_train_text_id_to_image_topk.csv
 - predictions_val_topk_four_directions.csv
 - run_metadata.json
Cache train: /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/m1/outputs/baseline_pipeline_m1/cache_embeddings/train_9e350cdf3477e6c0.npz
Cache val  : /Users/glutto/Documents/InformaticsEngineering/DeepMultimodalLearning/Final Project/m1/outputs/baseline_pipeline_m1/cache_embeddings/val_82e5b0dd91f26904.npz
Waktu pipeline sejak sel encoding dimulai: 333.5 detik.


## Serah-terima fitur untuk eksperimen berikutnya

Embedding disimpan dengan key:
- E: embedding gambar;
- T_ID: embedding caption Indonesia;
- T_EN: embedding caption Inggris;
- image_ids dan files: urutan pasangan untuk menyamakan indeks.

Bentuk matriks pada tiap split adalah jumlah baris × 512. Contoh memuat cache dari berkas yang dicetak pada sel ekspor:

    with np.load("path/cache_train.npz", allow_pickle=False) as cache:
        E_train = cache["E"]
        T_ID_train = cache["T_ID"]
        T_EN_train = cache["T_EN"]
        train_ids = cache["image_ids"]

Validasi ulang bahwa urutan image_id cocok sebelum membandingkan eksperimen. Untuk laporan M1, gunakan metrics_val dan catat split, arah, jumlah kandidat, seed, perangkat, waktu encoding, serta atribusi dataset.

## Atribusi dataset

Data: SEA-VL Indonesian subset (KORIKA-AI/sea-vl_crowdsourcing_id), turunan SEACrowd/sea-vl_crowdsourcing. Cahyawijaya et al. (2025), “Crowdsource, Crawl, or Generate? Creating SEA-VL, a Multicultural Vision-Language Dataset for Southeast Asia,” ACL. Lisensi CC BY-SA 4.0: https://creativecommons.org/licenses/by-sa/4.0/.

Pemeriksaan notebook mencakup skema, keberadaan gambar, ID/file yang tumpang tindih antarsplit, dan caption persis yang duplikat. Pemeriksaan ini bukan audit kesamaan visual atau kesamaan semantik lengkap.